# Student Performance Prediction — Data Preprocessing

This notebook prepares the Student Performance dataset for regression modeling. The workflow focuses on reproducible preprocessing and on comparing two feature scenarios:

- **Full feature set:** includes `G1` and `G2`
- **Early-prediction feature set:** excludes `G1` and `G2`

The target variable is `G3`, the final student grade.


## Imports


In [14]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


## Load Data


In [15]:
DATA_PATH = Path("../data/raw/student-mat.csv")

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Shape: {df.shape}")
df.head()


Shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## Target and Feature Sets

`G3` is used as the regression target.

Two feature sets are retained for later comparison:

- `X_full`: all predictors except `G3`
- `X_early`: excludes `G1` and `G2` to evaluate performance without prior-period grades

The second scenario is useful for assessing how much predictive signal is available before the student's first- and second-period grades are known.


In [16]:
TARGET = "G3"

y = df[TARGET]

X_full = df.drop(columns=[TARGET])
X_early = df.drop(columns=["G1", "G2", TARGET])

print(f"Full feature set:  {X_full.shape}")
print(f"Early feature set: {X_early.shape}")


Full feature set:  (395, 32)
Early feature set: (395, 30)


## Train–Test Split

The early-prediction feature set is used as the primary preprocessing example. The split is performed before fitting any transformations so that preprocessing parameters are learned from the training data only.


In [17]:
X = X_early.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")


X_train: (316, 30)
X_test:  (79, 30)


## Feature Groups

Categorical and numerical columns are identified from the training set so that each group can receive an appropriate transformation.


In [18]:
categorical_features = (
    X_train.select_dtypes(include="object").columns.tolist()
)

numerical_features = (
    X_train.select_dtypes(include="number").columns.tolist()
)

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features:   {len(numerical_features)}")


Categorical features: 17
Numerical features:   13


## Preprocessing Pipeline

Numerical features are median-imputed and standardized. Categorical features are imputed with the most frequent category and one-hot encoded.

`handle_unknown="ignore"` ensures that unseen categories at inference time do not raise an error.


In [19]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## Apply Transformations

The preprocessor is fitted on the training data and then applied to the test data using the learned parameters.


In [20]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Original training shape:    {X_train.shape}")
print(f"Processed training shape:   {X_train_processed.shape}")
print(f"Processed test shape:       {X_test_processed.shape}")


Original training shape:    (316, 30)
Processed training shape:   (316, 56)
Processed test shape:       (79, 56)


## Transformed Feature Names

One-hot encoding expands categorical variables into indicator columns. Inspecting the resulting feature names is useful for later model interpretation.


In [21]:
feature_names = preprocessor.get_feature_names_out()

print(f"Number of transformed features: {len(feature_names)}")
feature_names[:20]


Number of transformed features: 56


array(['num__age', 'num__Medu', 'num__Fedu', 'num__traveltime',
       'num__studytime', 'num__failures', 'num__famrel', 'num__freetime',
       'num__goout', 'num__Dalc', 'num__Walc', 'num__health',
       'num__absences', 'cat__school_GP', 'cat__school_MS', 'cat__sex_F',
       'cat__sex_M', 'cat__address_R', 'cat__address_U',
       'cat__famsize_GT3'], dtype=object)

## Preprocessing Summary

The dataset is now represented through a reproducible preprocessing pipeline that:

- separates numerical and categorical transformations
- imputes missing numerical values with the median
- imputes missing categorical values with the most frequent category
- standardizes numerical variables
- one-hot encodes categorical variables
- fits transformations only on the training data

Two modeling scenarios have been retained for comparison. The **full feature set** includes previous grades (`G1`, `G2`), while the **early-prediction feature set** removes them to evaluate how well student performance can be estimated from the remaining demographic, social, and school-related variables.

The fitted `preprocessor` can be combined directly with regression estimators in the modeling stage to ensure that identical transformations are applied during training and evaluation.
